# State cannabis product recalls

Collect the products, dates and reasons published by 11 state regulators. Run the notebook from top to bottom in JupyterLab or VS Code using a Python 3.10+ kernel. Cell 1 installs packages in that kernel. The collection starts in **Run the collection**. California opens each portal detail page and Missouri reads several large workbooks, so allow several minutes.

Results go into **recall_outputs** under the working directory printed below. Each state has a **recalls.csv** and **recalls.xlsx**. The first worksheet and the CSV contain the same rows. **collection_summary.csv** shows coverage by state. A second download supplied with this notebook contains an example run using the agency records available on November 24, 2026.

## 1. Install packages

In [1]:
%pip install -q "requests>=2.28" "beautifulsoup4>=4.12" "pandas>=2.0" "openpyxl>=3.1" "xlrd>=2.0" "pymupdf>=1.24.11" "xlsxwriter>=3.1"

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


## 2. Select states and the output folder

The default runs all 11 states. To try one first, set `STATES_TO_RUN = ["Michigan"]`. To save source documents alongside the tables, set `SAVE_DOCUMENTS = True`.

In [1]:
from pathlib import Path
from io import BytesIO
from urllib.parse import urljoin, urlsplit, parse_qs, unquote, quote
from datetime import datetime, timezone
from collections import OrderedDict
import csv
import hashlib
import json
import re
import shutil
import time
import unicodedata

import pandas as pd
import requests
import pymupdf
from bs4 import BeautifulSoup, Tag
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from IPython.display import display

SITES = {
    "California": "https://recalls.cannabis.ca.gov/",
    "Colorado": "https://med.colorado.gov/health-and-safety-advisory",
    "Missouri": "https://health.mo.gov/business-professionals/cannabis-regulation/licensee-compliance-guidance/product-recalls",
    "Maine": "https://www.maine.gov/dafs/ocp/resources/recalls",
    "Michigan": "https://www.michigan.gov/cra/bulletins/public-health-and-safety-advisory-bulletins-recalls",
    "Mississippi": "https://www.mmcp.ms.gov/compliance/compliance-bulletins-announcements",
    "Montana": "https://revenue.mt.gov/card/cannabis/cannabis-product-recalls",
    "Nevada": "https://ccb.nv.gov/guidance/#item-1",
    "New Jersey": "https://www.nj.gov/cannabis/adult-personal/recalls/",
    "New Mexico": "https://www.rld.nm.gov/cannabis/data-news/cannabis-recalls/",
    "New York": "https://cannabis.ny.gov/recalls",
    "Ohio": "https://com.ohio.gov/divisions-and-programs/cannabis-control/product-safety",
    "Oklahoma": "https://oklahoma.gov/omma/recalls/embargoed-and-recalled-products.html",
    "Washington": "https://lcb.wa.gov/examiners/recalls",
}
STATES_TO_RUN = list(SITES)  # Example for a shorter run: ["Michigan", "New York"]
OUTPUT = Path("recall_outputs")
LOCAL_SOURCES = {}  # Example: {SITES["Colorado"]: Path("saved/co_index.html")}
SAVE_DOCUMENTS = False  # True also keeps the downloaded source files under each state's folder.
REQUEST_DELAY = 0.30   # seconds between downloads
CA_ARCHIVE = "https://www.cannabis.ca.gov/product-safety/recalls/cannabis-recalls-archive/"
ME_MOBILE = "https://mobile.maine.gov/dafs/ocp/resources/recalls"

In [2]:
print("Output location:", OUTPUT.resolve())
print("Selected states:", ", ".join(STATES_TO_RUN))

Output location: C:\Users\pegah\Downloads\recall_outputs
Selected states: California, Colorado, Missouri, Maine, Michigan, Montana, New Jersey, New Mexico, New York, Ohio, Washington


## 3. Record notices and keep their sources

Each notice gets a `recall_id`. The collector keeps distinct published product descriptions within a notice, including differences in published size, status or category when extracted. A product can appear in multiple recalls. Colorado sometimes identifies a harvest or production batch as the affected item. Those rows say **published batch description**. Where the agency gives only a broad description, the row says **product group** or **source excerpt**. A notice with an unresolved product stays visible.

The dataset includes regulator-labeled advisories, updates and product alerts. Filter `notice_type` and `product_status` for your research definition. A reason taken from agency prose can describe several lots together. `date_basis` identifies the date used. The notice link and product source link can differ when a separate spreadsheet or PDF lists the items.

In [3]:
def clean(value):
    if value is None:
        return ""
    return " ".join(unicodedata.normalize("NFKC", str(value)).replace("\u200b", "").split())

def key(value):
    return re.sub(r"[^a-z0-9]+", "", clean(value).lower())

def txt(tag):
    return clean(tag.get_text(" ", strip=True)) if tag else ""

def dom(data):
    # UTF-8 is used by the agency sites. Older pages may declare another encoding.
    try:
        data = data.decode("utf-8-sig")
    except (UnicodeDecodeError, AttributeError):
        pass
    return BeautifulSoup(data, "html.parser")

def content(h):
    return h.find("article") or h.find("main") or h.find(id="main-content") or h

def anchors(node, base):
    return [(txt(a), urljoin(base, a["href"])) for a in node.select("a[href]")
            if not a["href"].startswith(("#", "mailto:", "javascript:"))]

def unique(values):
    return list(dict.fromkeys(x for x in values if x))

def digest(value):
    return hashlib.sha256(value.encode()).hexdigest()[:14]

MONTH = r"(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sept?(?:ember)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)"
DATES = re.compile(rf"\b(?:20\d{{2}}-\d{{1,2}}-\d{{1,2}}|\d{{1,2}}[-/]\d{{1,2}}[-/](?:20\d{{2}}|\d{{2}})|{MONTH}\.?,?\s+\d{{1,2}},?\s+20\d{{2}})\b",re.I)
def date_in(value):
    for m in DATES.finditer(clean(value)):
        raw = re.sub(r"\bSept\b","Sep",m[0],flags=re.I).replace(",","").replace(".","")
        for fmt in ("%Y-%m-%d","%m/%d/%Y","%m-%d-%Y","%m/%d/%y","%m-%d-%y","%B %d %Y","%b %d %Y"):
            try:
                return datetime.strptime(raw,fmt).date().isoformat()
            except ValueError:
                continue
    return ""

CAUSE = re.compile(r"resembles? non.marijuana|not the same packaging|as a result of|not properly tracked|banned |asbestos|unsafe levels|unapproved testing|due to|because|not (?:been )?(?:tested|compliant|approved|listed|certified)|failed|did not (?:pass|submit|meet)|not compliantly|unreliable|reported incorrectly|discovered|contamin|exceed|above (?:the )?(?:acceptable|action|limit|pass)|mislabel|misbrand|mislabell|undeclared|adulterat|testing requirements|incorrect|without (?:the )?(?:required|testing)|contained|contains?|adverse reaction|improper|pesticide|structural|chemical modification", re.I)
def cause_excerpt(text):
    sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z])|\n\s*\n", text)
    selected = [clean(s) for s in sentences if CAUSE.search(s) and 14 < len(clean(s)) < 1800
                and not re.search(r"if you|should contact|who experience|for example|may include|no adverse|no (?:reports|health.related)|are reminded|inhaling cannabis containing|may cause|if (?:a|the) (?:patient|consumer)",s,re.I)]
    return " ".join(unique(selected)[:3])

COLUMNS = ["state","recall_id","recall_date","product_name","recall_reason","record_level",
           "notice_type","source_section","product_status","company","product_type",
           "date_basis","notice_title","notice_url","product_source","pdf_page","detail","review"]
AUDIT = ["state","kind","url","result","detail"]
def new_session():
    s=requests.Session()
    s.headers["User-Agent"]="CannabisRecallResearch/1.1 (public notices)"
    retry=Retry(total=2,backoff_factor=.6,status_forcelist=[429,500,502,503,504],allowed_methods=["GET"])
    s.mount("https://",HTTPAdapter(max_retries=retry))
    return s

class Collection:
    def __init__(self,state):
        self.state=state
        self.session=new_session()
        self.events={}
        self.products={}
        self.log=[]
        self.documents={}
        self.folder=OUTPUT/state.lower().replace(" ","_")

    def audit(self,kind,url,result,detail=""):
        self.log.append(dict(zip(AUDIT,[self.state,kind,url,result,clean(detail)])))

    def get(self,url,kind,required=True):
        try:
            if url in self.documents:
                return self.documents[url]
            if url in LOCAL_SOURCES:
                data=Path(LOCAL_SOURCES[url]).read_bytes()
                source="saved copy"
            else:
                time.sleep(REQUEST_DELAY)
                response=self.session.get(url,timeout=(10,45))
                response.raise_for_status()
                data=response.content
                source="downloaded"
            if not data:
                raise ValueError("Empty response")
            if SAVE_DOCUMENTS:
                suffix=Path(unquote(urlsplit(url).path)).suffix.lower()
                if data.lstrip().startswith(b"%PDF"): suffix=".pdf"
                if not re.fullmatch(r"\.[a-z0-9]{1,6}",suffix): suffix=".html"
                path=self.folder/"sources"/(digest(url)+suffix)
                path.parent.mkdir(parents=True,exist_ok=True)
                path.write_bytes(data)
                source += " • "+path.name
            self.audit(kind,url,"ok",source)
            self.documents[url]=data
            return data
        except (OSError,requests.RequestException,ValueError) as exc:
            self.audit(kind,url,"review" if required else "unavailable",str(exc))
            return b""

    def event(self,identity,date,title,url,*,reason="",scope="",company="",kind="recall",section="",basis="agency notice date"):
        rid=self.state.upper().replace(" ","")[:4]+"-"+digest(identity)
        if rid not in self.events:
            self.events[rid]=dict(state=self.state,recall_id=rid,recall_date=date,recall_reason=reason,
                                  notice_title=title,notice_url=url,company=company,notice_type=kind,
                                  source_section=section,date_basis=basis,scope=scope)
        return self.events[rid]

    def add(self,e,name,source,*,level="named product",status="listed",category="",page="",detail=""):
        name=clean(name)
        if not name:return
        k=(e["recall_id"],name.casefold(),status,level,clean(category).casefold(),clean(detail))
        if k not in self.products:
            self.products[k]={"recall_id":e["recall_id"],"product_name":name,"record_level":level,
                              "product_status":status,"product_type":clean(category),"product_source":source,
                              "pdf_page":str(page),"detail":clean(detail)}

    def frames(self):
        rows=[]
        represented={p["recall_id"] for p in self.products.values()}
        for e in self.events.values():
            if e["recall_id"] not in represented:
                level="product group" if e["scope"] else "notice only"
                self.products[(e["recall_id"],"fallback")] = dict(recall_id=e["recall_id"],
                    product_name=e["scope"],record_level=level,product_status="listed",product_source=e["notice_url"])
        for p in self.products.values():
            e=self.events[p["recall_id"]]
            row={**e,**p}
            if not row.get("product_type"):row["product_type"]=e.get("pdf_scope","")
            problems=[label for field,label in [("recall_date","date missing"),("recall_reason","reason missing"),
                      ("product_name","product missing")] if not row.get(field)]
            if row["record_level"] in {"product group","notice only","source excerpt"}:problems.append("verify product scope")
            row["review"]=" | ".join(problems)
            rows.append(row)
        data=pd.DataFrame(rows,columns=COLUMNS).fillna("")
        if len(data):data=data.sort_values(["recall_date","recall_id","product_name"],ascending=[False,True,True],kind="stable")
        return data, pd.DataFrame(self.log,columns=AUDIT).drop_duplicates()

### Product tables and spreadsheets

Select columns by their headers, preserve product identifiers as text and collapse repeated listings of the same product. The source file retains the individual lot, package and retailer records.

In [4]:
from openpyxl import load_workbook

NAME_KEYS = ["productname","productdescription","product","item","itemname","productstrain",
             "strainname","strain","flavor","brandproduct","productbrand","itemdescription","name"]
TYPE_KEYS = ["producttype","producttypes","productcategory","category"]
def idx(headers, wanted):
    return next((headers.index(k) for k in wanted if k in headers),None)

def grid(table):
    """Preserve column positions when the publisher merges table cells."""
    pending={}
    for tr in table.select("tr"):
        if tr.find_parent("table") is not table:continue
        cols={i:value for i,(remaining,value) in pending.items()}
        pending={i:(n-1,v) for i,(n,v) in pending.items() if n>1}
        i=0
        for td in tr.find_all(["th","td"],recursive=False):
            while i in cols:i+=1
            value=txt(td)
            for j in range(max(1,int(td.get("colspan",1) or 1))):
                cols[i+j]=value
                span=int(td.get("rowspan",1) or 1)
                if span>1: pending[i+j]=(span-1,value)
            i+=max(1,int(td.get("colspan",1) or 1))
        if cols:yield [cols.get(j,"") for j in range(max(cols)+1)]

def read_named_rows(c,e,rows,url,*,status="listed",page="",fallback_type="",names=NAME_KEYS):
    """Read product columns only. Ignore retailer names, package IDs and blank cells."""
    headers=None
    found=0
    for row in rows:
        if headers is None:
            labels=[key(v) for v in row]
            name_ix=idx(labels,names)
            if name_ix is None:continue
            headers=labels
            type_ix=idx(labels,TYPE_KEYS)
            size_ix=idx(labels,["productsize","packagesize"])
            continue
        name=clean(row[name_ix]) if name_ix<len(row) else ""
        if not name or key(name) in names or key(name) in {"na","none","total","grandtotal"}:continue
        category=clean(row[type_ix]) if type_ix is not None and type_ix<len(row) else fallback_type
        size=clean(row[size_ix]) if size_ix is not None and size_ix<len(row) else ""
        c.add(e,name,url,category=category,page=page,status=status,detail=("Size: "+size) if size else "")
        found+=1
    return found

def spreadsheet(c,e,url,status="listed",category=""):
    data=c.get(url,"product list")
    if not data:return 0
    count=0
    try:
        if data[:4]==b"PK\x03\x04":
            book=load_workbook(BytesIO(data),read_only=True,data_only=True)
            try:
                for sh in book.worksheets:
                    if re.search(r"retail locations|dispensary locations",sh.title,re.I):continue
                    count+=read_named_rows(c,e,sh.values,url,status=status,fallback_type=category)
            finally:book.close()
        elif data.startswith(b"\xd0\xcf"):
            for df in pd.read_excel(BytesIO(data),sheet_name=None,header=None,dtype=str).values():
                count+=read_named_rows(c,e,df.fillna("").itertuples(index=False,name=None),url,status=status,fallback_type=category)
        else:
            if data.lstrip().startswith((b"<",b"%PDF")):raise ValueError("Expected a spreadsheet")
            try:s=data.decode("utf-8-sig")
            except UnicodeDecodeError:s=data.decode("cp1252")
            count=read_named_rows(c,e,csv.reader(s.splitlines(keepends=True)),url,status=status,fallback_type=category)
        if not count:c.audit("product list",url,"review","No named product column or rows. The file may list only identifiers.")
    except Exception as exc:c.audit("product list",url,"review",str(exc))
    return count

def section_nodes(heading):
    for tag in heading.find_all_next(True):
        if re.fullmatch(r"h[1-6]",tag.name) and int(tag.name[1])<=int(heading.name[1]):break
        yield tag

## 4. California portal and archive

The collector follows **View All Recalls** and each **Next** link, opens the notice, and reads its embedded product-details CSV where available. It reads the pre-2024 archive separately. California’s business recall date takes precedence over the DCC publication date when both appear.

In [5]:
CA_CARD=re.compile(r"/recalls/([0-9a-f]{24})(?:[/?#]|$)",re.I)
def ca_detail(c,e,url,card_name):
    data=c.get(url,"portal notice")
    if not data:return
    h=dom(data)
    fields={key(txt(dt)):txt(dt.find_next_sibling("dd")) for dt in h.find_all("dt")}
    e["scope"]=fields.get("productdescription") or card_name
    e["company"]=fields.get("legalbusinessname","")
    business=date_in(fields.get("businessrecalldate",""))
    pub=date_in(fields.get("dccrecallpublicationdate",""))
    if business or pub:
        e.update(recall_date=business or pub,date_basis="business recall date" if business else "DCC publication date")
    e["notice_type"]=fields.get("recalltype") or e["notice_type"]
    parts=[]
    for title in ["recallreason","recalloverview"]:
        tag=next((t for t in h.find_all(["h2","h3"]) if key(txt(t))==title),None)
        if tag:
            sibling=tag.find_next_sibling()
            if sibling and sibling.name not in {"h2","h3"}:parts.append(txt(sibling))
    if parts:e["recall_reason"]=" ".join(unique(parts))
    heading=next((t for t in h.find_all(["h2","h3"]) if key(txt(t))=="productdetails"),None)
    n=0
    if heading:
        for item in section_nodes(heading):
            if item.name=="table":n+=read_named_rows(c,e,grid(item),url)
    files=[u for t,u in anchors(h,url) if ".csv" in u.lower() and "location" not in (t+u).lower()]
    # The portal embeds the download URL in its server-rendered Next.js data.
    for tag in h.find_all("script",src=False):
        m=re.search(r"self\.__next_f\.push\((\[.*\])\)\s*;?\s*$",tag.get_text(),re.S)
        if not m:continue
        try:payload=json.loads(m[1])[-1]
        except (ValueError,IndexError):continue
        if not isinstance(payload,str):continue
        for match in re.finditer(r'"details"\s*:\s*\{',payload):
            try:obj,_=json.JSONDecoder().raw_decode(payload[match.end()-1:])
            except ValueError:continue
            if isinstance(obj.get("url"),str):files.append(urljoin(url,obj["url"]))
    for link in unique(files):
        n+=spreadsheet(c,e,link,category=fields.get("producttype",""))
    if not n and e["scope"]:
        c.add(e,e["scope"],url,level="product group",category=fields.get("producttype",""))
        c.audit("portal notice",url,"review","Product description retained. No more detailed product rows were available.")

def california(c):
    home=SITES[c.state]
    h=dom(c.get(home,"portal home",required=False))
    url=next((u for t,u in anchors(h,home) if "view all recalls" in t.lower()),urljoin(home,"recalls"))
    visited=set()
    seen=set()
    while url and url not in visited:
        visited.add(url)
        if len(visited)%10==0:print(f"  California portal page {len(visited)} …",flush=True)
        page=dom(c.get(url,"portal index"))
        cards={}
        for a in page.find_all("a",href=CA_CARD):
            href=urljoin(url,a["href"])
            box=a.find_parent(class_=lambda s:s and "shadow" in s) or a.parent
            title=txt(box.find(["h2","h3","h4"])) or txt(a)
            if title and title.lower()!="details":cards[href]=(title,box)
        new=set(cards)-seen
        if not new:
            c.audit("portal index",url,"review","No new recall cards on a requested page.")
            break
        for link,(name,box) in cards.items():
            if link in seen:continue
            seen.add(link)
            words=list(box.stripped_strings)
            reason=next((w for w in words if CAUSE.search(w) and w!=name),"")
            e=c.event(link,date_in(txt(box)),name,link,scope=name,reason=reason,section="live portal",
                      kind=next((w for w in words if key(w) in {"mandatory","voluntary"}),"recall"),
                      basis="portal listing date")
            ca_detail(c,e,link,name)
        next_url=next((u for t,u in anchors(page,url) if t.lower()=="next"),"")
        counter=re.search(r"Page\s+(\d+)\s+of\s+(\d+)",txt(page),re.I)
        if counter and int(counter[1])<int(counter[2]) and not next_url:
            c.audit("portal index",url,"review","The page count indicates another page, but its link was missing.")
        if next_url in visited:c.audit("portal index",next_url,"review","Pagination loop")
        url=next_url
    h=dom(c.get(CA_ARCHIVE,"archive index"))
    count=0
    for table in h.select("table"):
        rows=iter(grid(table));cols=[key(x) for x in next(rows,[])]
        if not {"recalldate","productdescription","recallreason"}<=set(cols):continue
        for row in rows:
            r=dict(zip(cols,row));name=r.get("productdescription","")
            if not name:continue
            ident=CA_ARCHIVE+"|"+r["recalldate"]+"|"+name
            e=c.event(ident,date_in(r["recalldate"]),name,CA_ARCHIVE,reason=r.get("recallreason",""),
                      company=r.get("legalbusinessname",""),section="pre-2024 archive",
                      kind=next((k for k in ["voluntary","mandatory"] if k in r.get("recalloverview","").lower()),"recall"))
            c.add(e,name,CA_ARCHIVE,level="archive description",category=r.get("producttype",""))
            count+=1
    if not count:c.audit("archive index",CA_ARCHIVE,"review","No recall rows matched the archive table headers.")

## 5. Missouri, Maine, New Jersey, New Mexico and New York

Missouri’s current and archived sections have different heading formats. The collector associates each product list with its notice and labels lists of removed products. Maine’s index identifies strains and flavors. Its individual notices supply reasons and sometimes a more detailed table. The official mobile page provides a second route to those records.

New Jersey supplies the reason in the product table. New Mexico links product lists in PDF and spreadsheet formats. For New York the collector resolves **current** and **past** separately and uses the *Issued* date. Lots removed in later updates carry a separate status.

In [6]:
def missouri(c):
    url=SITES[c.state]
    h=dom(c.get(url,"current and archive index"))
    current=None
    section=""
    notices=[]
    header=re.compile(r"^(\d{1,2}/\d{1,2}/\d{2,4})\s*[-–—]\s*([A-Z]{3}\d{6})\s*,\s*(.+)$")
    for t in h.find_all(["h2","h3","h4","div","button","p","strong","a"]):
        if t.name=="div" and t.get("role")!="button" and not t.string:continue
        s=txt(t)
        if key(s) in {"currentproductrecalls","archivedproductrecalls"}:
            section="current" if key(s).startswith("current") else "archive"
            current=None
            continue
        if not section:continue
        m=header.match(s) if len(s)<260 else None
        if m:
            ident=section+"|"+s
            if not current or current["identity"]!=ident:
                current=dict(identity=ident,date=date_in(m[1]),company=m[3],title=s,section=section,prose=[],links=[])
                notices.append(current)
        elif current and t.name=="p" and re.search(r"recalled|recall|did not|not tested|testing|cannabinoid",s,re.I):
            current["prose"].append(s)
        elif current and t.name=="a" and t.get("href") and re.search(r"list|notice of product recall",s,re.I):
            current["links"].append((s,urljoin(url,t["href"])))
    for a in notices:
        links=unique(a["links"])
        notice=next((u for t,u in links if "notice" in t.lower()),url)
        context=" ".join(unique(a["prose"]))
        e=c.event(a["identity"],a["date"],a["title"],notice,reason=cause_excerpt(context),company=a["company"],section=a["section"])
        lists=[(t,u) for t,u in links if "list" in t.lower() and re.search(r"\.xlsx?|\.csv",u,re.I)]
        for t,u in lists:
            spreadsheet(c,e,u,"removed from recall" if "removed" in t.lower() else "listed")
        if not lists:c.audit("product list",notice,"review","No linked product spreadsheet was recognized.")
        if not e["recall_reason"] and notice!=url:
            inspect_notice(c,e,notice)
    if not notices:c.audit("index",url,"review","No dated Missouri licensee headings matched.")

def maine(c):
    url=SITES[c.state]
    h=dom(b"")
    for address in [url,ME_MOBILE]:
        parent=dom(c.get(address,"recall index",False))
        candidates=[(address,parent)]
        for frame in parent.select("iframe"):
            if frame.get("srcdoc"):
                candidates.append((address,dom(frame["srcdoc"])))
            elif frame.get("src"):
                child=urljoin(address,frame["src"])
                if re.search(r"recall|advisory|table",child+" "+str(frame.get("title","")),re.I):
                    candidates.append((child,dom(c.get(child,"embedded recall table",False))))
        match=next(((u,h) for u,h in candidates if any("dateissued" in key(txt(t.find("tr"))) for t in h.select("table"))),None)
        if match:
            url,h=match
            break
    for table in h.select("table"):
        trs=table.select("tr")
        if not trs or "dateissued" not in key(txt(trs[0])):continue
        for tr in trs[1:]:
            cells=tr.find_all(["td","th"],recursive=False)
            if len(cells)<6:continue
            date,updated,kind,company,category,flavors=map(txt,cells[:6])
            links=anchors(tr,url)
            link=links[0][1] if links else url
            if url==ME_MOBILE:link=link.replace("://www.maine.gov/","://mobile.maine.gov/")
            e=c.event(link+"|"+date,date_in(date),f"{kind} – {company}",link,company=company,kind=kind,section="recalls and advisories")
            d=dom(c.get(link,"notice")) if link!=url else None
            n=0
            if d:
                main=content(d)
                paras=[txt(p) for p in main.find_all("p")]
                e["recall_reason"]=cause_excerpt(" ".join(paras[:15]))
                for dt in main.select("table"):
                    n+=read_named_rows(c,e,grid(dt),link,names=["strainname","strain","productname","flavor","strainsflavors","product"])
            # Include each index strain even when a detail table lists only some strains.
            for name in unique([clean(x) for x in re.split(r"[,;]",flavors)]):
                if key(name) in {"na","multiple","various","seeadvisory"}:continue
                if not any(p["recall_id"]==e["recall_id"] and key(p["product_name"])==key(name) for p in c.products.values()):
                    c.add(e,name,url,category=category)
            e["scope"]=category
    if not c.events:c.audit("index",url,"review","No recall rows found. For an iframe, save its table page and map it to the index URL.")

def new_jersey(c):
    url=SITES[c.state]
    h=dom(c.get(url,"recall index"))
    for table in h.select("table"):
        rows=iter(grid(table));cols=[key(x) for x in next(rows,[])]
        if "productname" not in cols:continue
        for row in rows:
            r=dict(zip(cols,row))
            name=r.get("productname","")
            if not name:continue
            date=date_in(r.get("announcementdate",""))
            company=r.get("manufacturer","")
            e=c.event("|".join([url,date,company,name]),date,name,url,reason=r.get("reasonforrecall",""),
                      company=company,section="published table",basis="announcement date")
            c.add(e,name,url,detail=r.get("updates","") if key(r.get("updates",""))!="na" else "")
    if not c.events:c.audit("index",url,"review","No product rows matched the New Jersey table.")

def new_mexico(c):
    url=SITES[c.state]
    h=dom(c.get(url,"recall index"))
    for t in h.select("table"):
        rows=iter(grid(t));cols=[key(x) for x in next(rows,[])]
        if not {"recalldate","productname","recallreason"}<=set(cols):continue
        title=txt(t.find_previous(["h3","h4","h5","h6"]))
        trs=t.select("tr")[1:]
        for tr,row in zip(trs,rows):
            r=dict(zip(cols,row));name=r["productname"]
            scope=re.split(r"see recall details",name,flags=re.I)[0].strip(" .")
            links=anchors(tr,url)
            notice=next((u for t,u in links if key(t)=="recallnotice"),links[0][1] if links else url)
            e=c.event(url+"|"+r["recalldate"]+"|"+scope,date_in(r["recalldate"]),title,notice,
                      reason=r["recallreason"],scope=scope,kind="mandatory recall" if "mandatory" in title.lower() else "recall",section="published table")
            for label,src in links:
                if re.search(r"list of|package ids",label,re.I):
                    if ".pdf" in src.lower():pdf_products(c,e,src,attachment=True)
                    else:spreadsheet(c,e,src,category=r.get("producttype","").split("See recall")[0])
            if not any(p["recall_id"]==e["recall_id"] for p in c.products.values()):
                c.add(e,scope,url,level="product group",category=r.get("producttype","").split("See recall")[0])
    if not c.events:c.audit("index",url,"review","No dated tables matched.")

def new_york(c):
    landing=SITES[c.state]
    h=dom(c.get(landing,"landing page",False))
    for section in ("current","past"):
        expected=r"cur+ent" if section=="current" else r"past"
        url=next((u for t,u in anchors(h,landing) if re.search(expected+r".*recalls",t,re.I)),urljoin(landing,"/"+section+"-recalls"))
        data=c.get(url,section+" index")
        if not data:continue
        page=dom(data)
        root=page.find("main") or page
        count=0
        for heading in root.find_all(["h2","h3"]):
            if key(txt(heading)) in {"overview","onthispage","currentrecalls","pastrecalls"}:continue
            elems=list(section_nodes(heading))
            paragraphs=[txt(tag) for tag in elems if tag.name=="p" and not tag.find_parent("table")]
            body=" ".join(paragraphs)
            issued=re.search(r"\bIssued\s*:\s*("+DATES.pattern+r")",body,re.I)
            if not issued:continue
            title=txt(heading)
            date=date_in(issued[1])
            e=c.event(title+"|"+date,date,title,url,reason=cause_excerpt(body.split(issued[0],1)[-1]),
                      company=title,section=section,basis="issued date")
            count+=1
            for table in (el for el in elems if el.name=="table"):
                read_named_rows(c,e,grid(table),url)
            for li in (el for el in elems if el.name=="li"):
                m=re.search(r";\s*([^;]+?)\s*;\s*Lot\s*#?\s*:\s*(.+)",txt(li),re.I)
                if m and re.search(r"remov",body.split(issued[0],1)[0],re.I):
                    c.add(e,m[1],url,status="removed from recall",detail="Lot "+m[2])
        if not count:
            has_overview=root.find(string=re.compile(r"products currently subject|recalls that have been completed",re.I))
            only_overview=has_overview and not root.select("table") and not re.search(r"\bIssued\s*:",txt(root),re.I)
            c.audit(section+" index",url,"no listings displayed" if only_overview else "review",
                    "Index opened and displayed no notice sections." if only_overview else "Check for a changed notice layout.")

## 6. Read products inside bulletins and PDFs

The parser reads PDF text and tables across pages. It recognizes Michigan package-tag lists, Colorado harvest-batch headings, Montana product lists and Ohio product IDs. PDF page numbers are one-based. Image-only files, undetected product columns and documents with complex lists are flagged in **Sources**. For a scanned source you can supply a searchable PDF under its original URL.

In [7]:
MI_TAG=r"1A[0-9A-Z]{22}"
def mi_product_lines(c,e,body,url,number):
    """Michigan bulletins frequently put an item immediately after its package tag."""
    raw=[l.strip() for l in body.splitlines() if clean(l)]
    for i,line in enumerate(raw):
        s=clean(line)
        after=re.search(r"(?:Individual\s+)?(?:Package|Metrc)\s*#\s*"+MI_TAG+r"\s*(.*)$",s,re.I)
        name=""
        if after:
            tail=after[1].strip(" ()–—-")
            if tail and not date_in(tail):name=tail
            elif not tail and i+1<len(raw):
                candidate=clean(raw[i+1])
                if not re.search(r"^sold|^public health|^package\b|^1A4|^this (?:bulletin|advisory)|^license|^\d{1,2}/",candidate,re.I):
                    name=candidate
        else:
            first=re.match(r"^(.+?)\s*[-–—]\s*"+MI_TAG+r"\b",s)
            if first:name=first[1]
        if name and len(name)<160 and re.search("[A-Za-z]",name) and not re.search(r"license|^sold|\bLLC\b|^(?:patients|consumers|patients or caregivers)",name,re.I):
            c.add(e,name,url,page=number)
        if re.match(MI_TAG,s,re.I) and i:
            before=raw[i-1]
            # Two-column flower lists align names immediately above the two tag numbers.
            if not re.search(MI_TAG+r"|address|facility|license|^\d|(?:Street|Road|Avenue)\b|^sold |\bMI\s+\d|:|public health|bulletin|^Page\b",before,re.I):
                for field in re.split(r"\s{3,}",before):
                    item=re.split(r"\s*[-–]\s*Vitamin E detected",clean(field),flags=re.I)[0]
                    if item and len(item)<72 and re.search("[A-Za-z]",item) and not date_in(item):
                        c.add(e,item,url,page=number)
        for field in re.split(r"\s{3,}",line):
            item=clean(field)
            if re.fullmatch(r".{2,65}\s*\((?:Flower|Concentrate|Vape|Edible)\)",item,re.I):
                c.add(e,item,url,page=number)
    paragraph=clean(body)
    for pattern in [r"Consumers who purchased (.{3,110}?) produced by",
                    r"A single batch of (.{3,110}?) is being recalled",
                    r"product is spelled [“\"]([^”\"]+)",
                    r"manufactured under the name [“\"]([^”\"]+)",
                    r"Most were labeled as [“\"]([^”\"]+)",
                    r"(?:affected marijuana products are marketed under the brand) [“\"]([^”\"]+)"]:
        match=re.search(pattern,paragraph,re.I)
        if match:c.add(e,match[1].strip(" ."),url,page=number)
    if "Mary's" in body:
        # Product headings in the Choice Labs bulletin precede columns of dispensary names.
        for item in raw:
            if re.search(r"Mary.s",item,re.I) and re.search(r"patch|tincture|compound|gel|capsule",item,re.I) and len(item)<85:
                c.add(e,item,url,page=number)

    if number==1:
        if "Lead Contamination in Vape Cartridges" in paragraph:e["scope"]="Vape cartridges with lead contamination"
        for pattern in [r"(products tested by Iron Laboratories)",r"(?:products? (?:were|was) tested (?:at|by) The Spott)",r"(packages of pre-rolls) after the packages had been sampled",r"(?:recalling [\d,]+ units of )(.{4,60}? produced by [^(]{3,55})\s*\("]:
            m=re.search(pattern,paragraph,re.I)
            if m:e["scope"]=clean(m[1] if m.lastindex else m[0])

In [8]:
import warnings
from urllib.parse import urlencode

def file_url(url):
    host=urlsplit(url).hostname or ""
    m=re.search(r"/(?:file|document)/d/([-\w]+)",url)
    if host=="docs.google.com" and m:
        return f"https://docs.google.com/document/d/{m[1]}/export?format=pdf"
    if host=="drive.google.com":
        fileid=m[1] if m else parse_qs(urlsplit(url).query).get("id",[""])[0]
        if fileid:return "https://drive.google.com/uc?"+urlencode({"export":"download","id":fileid})
    return url

def pdf_date(lines):
    for line in lines[:28]:
        if re.search(r"sold|sale|purchase|tested|manufactur|received|distributed",line,re.I):continue
        if DATES.match(line) or re.match(r"^(?:DATE|ISSUED|RELEASE DATE|FOR IMMEDIATE RELEASE)\s*:?",line,re.I):
            d=date_in(line)
            if d:return d
    return ""

BATCH_HEAD=re.compile(r"^(?:\*?added\s+)?(?:(?:potentially\s+)?contaminated|affected|recalled|untested|unapproved|potentially unsafe)?\s*(?:harvest|production|product)?\s*batch(?:es| numbers?| #s?)?\s*(?::|[-–]|$)",re.I)
BATCH_STOP=re.compile(r"^(?:for (?:more|questions)|.*(?:dispensaries|stores|locations) (?:that|where)|(?:retail|medical) (?:marijuana|license)|license (?:number|name)|this (?:product|advisory)|consumers? |all regulated|doing business|dba\b|address\b|source:|contact\b)",re.I)
def pdf_products(c,e,url,attachment=False):
    target=file_url(url)
    # A saved copy can be keyed by the original agency link.
    data=c.get(url if url in LOCAL_SOURCES else target,"product attachment" if attachment else "notice file")
    if not data:return 0
    if not data.lstrip()[:10].startswith(b"%PDF"):
        c.audit("PDF",url,"review","Expected PDF bytes. Map the original URL to a saved PDF in LOCAL_SOURCES.")
        return 0
    before=len(c.products)
    try:
        with pymupdf.open(stream=data,filetype="pdf") as doc:
            pages=[unicodedata.normalize("NFKC",p.get_text(sort=True)).replace("\u200b","") for p in doc]
            if not any(len(clean(p))>=35 for p in pages):
                c.audit("PDF",url,"review","Image-only or unreadable text. Supply a searchable/OCR copy.");return 0
            first=pages[0]
            if not attachment:
                if not e["recall_date"]:e["recall_date"]=pdf_date([clean(l) for l in first.splitlines() if clean(l)])
                if not e["recall_reason"]:
                    # Join wrapped lines within paragraphs before finding the regulator's explanation.
                    paras=[clean(p) for p in re.split(r"\n\s*\n",first) if len(clean(p))>38]
                    if c.state=="Ohio":
                        issue=re.search(r"\bISSUE:\s*(.*?)(?=\n\s*\n|$)",first,re.I|re.S)
                        topic=re.search(r"\bTOPIC:\s*(.*?)\s+TYPE:",first,re.I|re.S)
                        e["recall_reason"]=cause_excerpt(" ".join(x for x in [clean(topic[1]) if topic else "",clean(issue[1]) if issue else ""] if x))
                    if not e["recall_reason"]:e["recall_reason"]=cause_excerpt(" ".join(paras[:13]))
                if c.state=="Colorado":
                    scope=re.search(r"\b(?:regulated marijuana|marijuana) (flower(?:\s*\([^)]{2,40}\))?|concentrates?|vapori[sz]er delivery devices?|infused products?)",clean(first),re.I)
                    if scope:e["pdf_scope"]=scope[0]
                    updated=re.search(r"\bUPDATED\s+("+DATES.pattern+r")",clean(first),re.I)
                    if not e["recall_date"] and updated:
                        e.update(recall_date=date_in(updated[1]),date_basis="advisory update date")
                if c.state=="Colorado" and re.search(r"(?:initiated?|conduct(?:ed|ing)?|issu(?:e|ed)|voluntary)\s+(?:a\s+)?(?:voluntary\s+)?recall",first,re.I):
                    e["notice_type"]="advisory reporting a recall"
            batch_section=False
            for number,(page,body) in enumerate(zip(doc,pages),1):
                lines=[clean(l) for l in body.splitlines() if clean(l)]
                if attachment or re.search(r"(?:product|item|strain)\s+name",body,re.I):
                    # Limit table work to pages likely to contain product data.
                    for tab in page.find_tables().tables:
                        raw=tab.extract()
                        count=read_named_rows(c,e,raw,url,page=number)
                        if not count and c.state=="New Mexico" and attachment:
                            # Some tables have a product heading above retailer/package columns.
                            for i,row in enumerate(raw[:-1]):
                                cells=[clean(x) for x in row if clean(x)]
                                below=" ".join(clean(x) for x in raw[i+1]).lower()
                                if len(cells)==1 and re.search(r"retail business|licensed address",below) and not re.search(r"retail|package id",cells[0],re.I):
                                    c.add(e,cells[0],url,page=number)
                if attachment:continue
                if c.state=="Colorado":
                    for raw in body.splitlines():
                        line=clean(raw)
                        named=re.match(r"^(?:Item|Product) Name:\s*(.+)",line,re.I)
                        if named:
                            c.add(e,named[1],url,page=number)
                            continue
                        if (BATCH_HEAD.search(line) or
                            (len(line)<100 and re.search(r"(?:harvest|production) batch(?:es| numbers?)\s*:$",line,re.I))):
                            batch_section=True
                            continue
                        if not line:continue
                        if re.search(r"\b(?:stores?|locations?|centers?) (?:where|that)|^(?:\*\*|sold (?:at|from)|affected stores|medical center|facility name|failed testing|distributed to)|(?:sold|available) (?:at|by|from)|(?:\bLLC|\bInc)\b|^Note\b",line,re.I) or len(line)>145 or BATCH_STOP.search(line) or re.search(r"\b(?:Street|Avenue|Blvd)\b|\b(?:40[234]R?-\d{5}|\d{5} CO)\b",line,re.I):
                            batch_section=False
                        if batch_section and len(line)<145 and not re.search(r"Page\s+\d|^\d+\s*(of|/)\s*\d+$|^https?:|^(?:[-_*]|\d\s*\|\s*Page)$",line,re.I):
                            for item in re.split(r"\s{3,}",raw.strip()):
                                if key(item) not in {"","productname","productionbatchnumber","harvestbatchnumber"} and not re.fullmatch(r"\d",clean(item)):
                                    c.add(e,item,url,level="published batch description",page=number)
                if c.state=="Michigan":
                    mi_product_lines(c,e,body,url,number)
                    for line in lines:
                        name=re.match(r"^Product\s+Name\s*:\s*(.+)",line,re.I)
                        if name:
                            c.add(e,re.split(r"\s+(?:Production Batch|Package|Metrc)\s*(?:#|number|tag|:)",name[1],flags=re.I)[0],url,page=number)
                    if number==1:
                        paragraph=" ".join(lines)
                        m=re.search(r"(?:impacted products include the following strains:|flavors affected by this voluntary recall include)\s*(.{8,600}?)(?:\.\s|$)",paragraph,re.I)
                        if m:
                            for name in re.split(r",\s*(?:and\s+)?|\s+and\s+",m[1]):c.add(e,name,url,page=number,category="strain or flavor")
                if c.state in {"Montana","Michigan"}:
                    in_list=False
                    for line in lines:
                        if re.search(r"items identified|following strains|following products|following flavors",line,re.I):
                            in_list=True
                        m=re.match(r"^[•●▪]\s*(.{2,140})$",line)
                        if m and in_list:c.add(e,m[1],url,page=number)
                        elif in_list and re.search(r"retail (?:location|store)|please |consumers who|license number",line,re.I):in_list=False
                if c.state=="Ohio":
                    for line in lines:
                        m=re.search(r"\b(M\d{8,})\s*:\s*(.+)",line)
                        if m:c.add(e,m[2],url,page=number,detail=m[1])
            if not attachment and not any(p["recall_id"]==e["recall_id"] for p in c.products.values()) and not e["scope"]:
                # An explicit product-scope sentence is retained with its original wording and page.
                candidates=[]
                for para in re.split(r"\n\s*\n",first):
                    s=clean(para)
                    if re.search(r"consumers who purchased|recall (?:of|for|on)|is recalling|is voluntarily recalling|affected marijuana products are|recall affects|due to (?:the identification|marijuana flower|the presence)|products? (?:affected|being recalled)|(?:vapes?|cartridges?|gummies|flower) (?:contained|produced|sold)|affected (?:products?|harvest batches) (?:include|are)",s,re.I) and len(s)<1400:
                        candidates.append(s)
                if e.get("pdf_scope"):e["scope"]=e["pdf_scope"]
                elif candidates:c.add(e,candidates[0],url,level="source excerpt",page=1)
            if len(c.products)==before or (not attachment and c.state in {"Michigan","Colorado","Ohio"}):
                c.audit("PDF extraction",url,"review","Check the notice for additional products, lot-only lists, continuation pages or linked appendices.")
    except Exception as exc:c.audit("PDF extraction",url,"review",str(exc))
    return len(c.products)-before

def inspect_notice(c,e,url):
    target=url if url in LOCAL_SOURCES else file_url(url)
    data=c.get(target,"notice document")
    if not data:return
    if data.lstrip().startswith(b"%PDF") or target!=url or urlsplit(url).path.lower().endswith(".pdf"):
        pdf_products(c,e,url)
        return
    h=dom(data);main=h.find("main") or h.find("article") or h
    for table in main.select("table"):
        read_named_rows(c,e,grid(table),url)
    lines=[txt(p) for p in main.select("p")]
    if not e["recall_reason"]:e["recall_reason"]=cause_excerpt(" ".join(lines[:16]))
    if not e["recall_date"]:
        m=re.search(r"\b(?:DATE|Issued)\s*:\s*("+DATES.pattern+")",txt(main),re.I)
        if m:e["recall_date"]=date_in(m[1])
    for label,link in unique(anchors(main,url)):
        if link!=url and re.search(r"\.pdf(?:\?|$)",link,re.I) and re.search(r"recall|affected product",label,re.I):
            pdf_products(c,e,link,attachment=bool(e["recall_reason"]))

### Colorado, Michigan, Montana and Ohio

These collectors start from the index on every run. If Colorado or Michigan does not serve the index, the collector tries previously identified agency bulletin links and marks the state **partial index**. Ohio tries two agency index addresses, then known DCC notices. One Ohio notice also has dated, group-level metadata for use when its product table is unavailable. These fallback lists have no guaranteed historical or current coverage.

In [9]:
CO_DOCUMENTS = [
  [
    "RMRC LLC, doing business as High Octane",
    "https://drive.google.com/file/d/1eNsw8cITFVIFlTvpF1KXNHiFP1jWxaJB/view"
  ],
  [
    "EXPANDED Double C Farms",
    "https://drive.google.com/file/d/1s5YrlVCaIsEywcYMU-ughAiXbQnLFe0V/view"
  ],
  [
    "EXPANDED HG LTD, doing business as Higher Grade",
    "https://drive.google.com/file/d/1WFrJsUtGrK0J9CibBjMAwK4i_kXtlpvK/view"
  ],
  [
    "Sweetwater Group LLC",
    "https://drive.google.com/file/d/13YLI15IZ7lGFeVAnZES8vw1jlCuMczC5/view"
  ],
  [
    "EXPANDED Karing Kind LLC",
    "https://drive.google.com/file/d/1pD-nht94Id2_tmp6-WDPiFNciH67jxVq/view"
  ],
  [
    "Dutch Botanicals LLC, doing business as Morganic Concentrates",
    "https://drive.google.com/file/d/1r3Tyzturym13Qc6BjpToStviBvOOtUwk/view"
  ],
  [
    "MBF Labs LLC, doing business as My brothers flower",
    "https://drive.google.com/file/d/1KvyOhoFc8TmE6_A6Tn_8_pEVBsfHd8Yc/view"
  ],
  [
    "HG LTD, doing business as Higher Grade",
    "https://drive.google.com/file/d/1GKnyLFccnLupVkDLLZwADODi7o8x40Cd/view"
  ],
  [
    "Egozi Ventures LLC",
    "https://drive.google.com/file/d/1hVBxJXY-_7ZXrCz3yI0J7CENF9eas4Dy/view"
  ],
  [
    "Rotation LLC",
    "https://drive.google.com/file/d/19q__x77AyU7ZTTrhGXGGMJNjWyJ9R5Sj/view"
  ],
  [
    "CC Brands",
    "https://drive.google.com/file/d/1RhmUmKfERHw3D39LuM5dmTCr80EZ5cQs/view"
  ],
  [
    "EXPANDED Karing Kind LLC",
    "https://drive.google.com/file/d/155K5itnbbcG9r6nxemU_8UHDu709cMJB/view"
  ],
  [
    "Double C Farms LLC",
    "https://drive.google.com/file/d/1UAGSwwkSjaNBMT4_55iNo-xjhiBuTt4H/view"
  ],
  [
    "1-11 LLC",
    "https://drive.google.com/file/d/1wxGzfS5E5CvpSdIl0p3_wpaUGOVlg7-O/view"
  ],
  [
    "Karing Kind LLC",
    "https://drive.google.com/file/d/1BcojfgROiT85OPH9lhVlaLJC1lCHPb4_/view"
  ],
  [
    "Phoenix Farmstead LLC",
    "https://drive.google.com/file/d/1d9dxbbcp38qhitq_nnBXNGXx8tihY-eu/view"
  ],
  [
    "RRSA LLC",
    "https://drive.google.com/file/d/1BHySC8XCeVLWUZPjUgiyYtlaRfJC5Pin/view"
  ],
  [
    "Checker Six LLC",
    "https://drive.google.com/file/d/1Fv-ODcgUshxPwdKn31ODct8y0ZZ4n_gN/view"
  ],
  [
    "CSV CO LLC and Jolet Ventures LLC",
    "https://drive.google.com/file/d/1PXcEnUUvJ0FhKHSGMPAdqc7xq6oSlZPJ/view"
  ],
  [
    "Clearview Industries LLC",
    "https://drive.google.com/file/d/1CiIaN9i_LaRfkPphezsddl7jz6QYva7p/view"
  ],
  [
    "C&R Colorado (dba Smokiez)",
    "https://drive.google.com/file/d/1uTORm-zZZvUK2YC_Tbh59h_0rcQZPjjT/view"
  ],
  [
    "Verv LLC (dba Dadirri)",
    "https://drive.google.com/file/d/1S-DzTLFKhdxm9scoC-EgZSoyHluy3oCm/view"
  ],
  [
    "UPDATED Magic City Farms 1 LLC",
    "https://drive.google.com/file/d/1XjvXw1cBy9RUUsRO95g9YQVBYN2aPKP4/view"
  ],
  [
    "PBS HOLDCO LLC, dba Purplebee's",
    "https://drive.google.com/file/d/1iAnXF7p_mCzCHRMVsxGmuYuGdA7lrBwR/view"
  ],
  [
    "Fox Street Wellness LLC",
    "https://drive.google.com/file/d/1-8eWEfr3zRc4GI3VXz5GTGJHl-Am0P1W/view"
  ],
  [
    "MedPharm Holdings LLC",
    "https://drive.google.com/file/d/1idBqkIEOYMygYTyIrTMjdJZgRSjlyX_O/view"
  ],
  [
    "Levels IV Inc and SKL Holdings III LLC",
    "https://drive.google.com/file/d/1_wFzm3GOJfzE6tmSqIhIcyvtI8ocZJDf/view"
  ],
  [
    "RR Services, LLC (dba Bloom County), 11/14/2024",
    "https://drive.google.com/file/d/1Ouz0dMZyAoBPFFfo9cKvW7RWSfsX92Ea/view"
  ],
  [
    "Confections Colorado LLC doing business as “Wyld,”",
    "https://drive.google.com/file/d/19WkOCu8hxtPrD_NbEIiARDM6r6TAv91B/view"
  ],
  [
    "Ware Hause LLC and South Platte Distributors LLC, 6/6/24",
    "https://drive.google.com/file/d/1rcQ8SVl7CWyI11LnNKLViIZCQJPg0e5d/view"
  ],
  [
    "DSA LLC, 3/6/24",
    "https://drive.google.com/file/d/17r2cgzx1nZo0qrQfl7FQkYJGhtyKaTK2/view"
  ],
  [
    "I & S LLC, 12/6/23",
    "https://drive.google.com/file/d/1e5yHNYegtpZHk-6_T6a0lbbod9VH066P/view"
  ],
  [
    "Boone Farms, LLC 11/16/23",
    "https://docs.google.com/document/d/1kiUXSn8BKV9WlDQQAOVe2se6oJ5K2fWF/edit?ouid=107710753235936513452&rtpof=true&sd=true&usp=sharing"
  ],
  [
    "Angel Farms, LLC 11/16/23",
    "https://docs.google.com/document/d/1VgJQwU33My2RDaXukQ8yGG66YFAu_OSH/edit?ouid=107710753235936513452&rtpof=true&sd=true&usp=sharing"
  ],
  [
    "JEM Dutch Acres, LLC (2/28/22)",
    "https://drive.google.com/file/d/1I-nbQQyv4UKLNghmjmbliC_WVa8fkpeJ/view"
  ],
  [
    "DJR Colorado, LLC (2/2/22)",
    "https://docs.google.com/document/d/1S_40b2SJe9L_ZeHc-fK7Klq19YMnDROTX3cy81itYks/edit"
  ],
  [
    "First Class LLC, November 18, 2021",
    "https://drive.google.com/file/d/1ap4Pkw4aJxhvzy9G0tbZNkViwE69_6Ry/view"
  ],
  [
    "Potentially Unsafe Levels of Microbial Contamination in Products from Several Businesses",
    "https://drive.google.com/open?id=1RJSbwmD5DzoI-E09vDshbUATjN9wvx6h"
  ]
]

MI_DOCUMENTS = [
  [
    "08-27-2025 - Notification of Marijuana Product Recall",
    "https://www.michigan.gov/cra/-/media/Project/Websites/cra/bulletin/1Public-Health-an-Safety-Advisory/Recall-Bulletin-Exclusive.pdf?rev=b6068d144c9c4f60916bc65cbddd28fe&hash=E73629C4D89E5D9CB2054303E987AF0B"
  ],
  [
    "08-12-2025 - Notification of Voluntary Marijuana Product Recall",
    "https://www.michigan.gov/cra/-/media/Project/Websites/cra/bulletin/1Public-Health-an-Safety-Advisory/Voluntary-Product-Recall-Sky-Cannabis-LLC.pdf?rev=6ec6d8bb28534bfabcf9789f3626bd47&hash=9FD60E26E367672DFECB8C19290565C4"
  ],
  [
    "02-11-2025 - Notification of Voluntary Product Recall",
    "https://www.michigan.gov/cra/-/media/Project/Websites/cra/bulletin/1Public-Health-an-Safety-Advisory/Exhale-Public-health-and-Safety-Bulletin-021125.pdf?rev=1c64c3ada02348a1aa1b7379d90732cc&hash=EF31B808D231E17B6815B0BEDE01CDCD"
  ],
  [
    "01-31-2025 - Notification of Voluntary Marijuana Product Recall",
    "https://www.michigan.gov/cra/-/media/Project/Websites/cra/bulletin/1Public-Health-an-Safety-Advisory/RWB-Public-Health-and-Safety-Bulletin-FINAL.pdf?rev=901c3e7565a648f3a5743ea4d152de5e&hash=D64B557089E3BA6F6DD17A0247AC0F1B"
  ],
  [
    "05-15-2024 - Notification of Marijuana Product Recall",
    "https://www.michigan.gov/cra/-/media/Project/Websites/cra/bulletin/1Public-Health-an-Safety-Advisory/Recall-Bulletin---Flavor-Galaxy---FINAL.pdf?rev=375f379c167d4c869b408433c4f84c07&hash=FB9BD1B86AC54873FDFD39F2E40EE452"
  ],
  [
    "05-15-2024 - Marijuana Consumer Advisory",
    "https://www.michigan.gov/cra/-/media/Project/Websites/cra/bulletin/1Public-Health-an-Safety-Advisory/Public-Health-and-Safety-Advisory---Flavor-Galaxy---FINAL.pdf?rev=f561784a72bc48dca7c4a2e46c98def7&hash=44C217F89379C3FA1051186768B59D57"
  ],
  [
    "11-20-2023 - Notification of Voluntary Marijuana Product Recall",
    "https://www.michigan.gov/cra/-/media/Project/Websites/cra/bulletin/1Public-Health-an-Safety-Advisory/Recall-Bulletin-112023---FINAL.pdf?rev=11a4f517c3c143cf84ab750e72fa4615&hash=A9CC2785F5A09C1040BE68E22AF1BE15"
  ],
  [
    "11-15-2023 - Marijuana Customer Advisory",
    "https://www.michigan.gov/cra/-/media/Project/Websites/cra/Final-Orders/2023/October/Public-Health-and-Safety-Advisory---Marijuana-Consumer-Alert--MI-10-FINAL.pdf?rev=727b2022030e4cc2884d8f5e9af420ef&hash=1B6B6BB7612425A08E338449C0ECDCF3"
  ],
  [
    "11-08-2023 - Notification of Voluntary Marijuana Product Recall",
    "https://www.michigan.gov/cra/-/media/Project/Websites/cra/bulletin/1Public-Health-an-Safety-Advisory/RecallBulletin110823FINAL.pdf?rev=2a522fd6d86f4d8db9ec870cf95727cc&hash=3EEACBC0C1857DBC440DFB729E954770"
  ],
  [
    "09-27-2023 - Notification of Voluntary Marijuana Product Recall",
    "https://www.michigan.gov/cra/-/media/Project/Websites/cra/bulletin/health-safety/Public-Health-and-Safety-Advisory---Voluntary-Recall---SBCMI_-FINAL.pdf?rev=090120d4002f412786bdc9e67659e7f9&hash=4FABF0FD2E4F375D5319D4C92C89162B"
  ],
  [
    "07-21-2023 - Notification of Voluntary Marijuana Product Recall",
    "https://www.michigan.gov/cra/-/media/Project/Websites/cra/bulletin/1Public-Health-an-Safety-Advisory/Public-Health-and-Safety-Advisory---Marijuana-Consumer-Alert--Flight-Vapes--FINAL.pdf?rev=1a69572d1e924df99afaa878cc8cdf22&hash=78E854F4A224534E09333D2B572B1432"
  ],
  [
    "07-07-2023 - Notification of Voluntary Marijuana Product Recall",
    "https://www.michigan.gov/cra/-/media/Project/Websites/cra/bulletin/1Public-Health-an-Safety-Advisory/Public-Health-and-Safety-Advisory---Voluntary-Recall---Weed-Bar_Final-(772023).pdf?rev=59f38109299547be81bcfc898b4c866c&hash=F1A6E832BB9C238BE5827F7C4527AE88"
  ],
  [
    "11-15-2022 - Marijuana Consumer Advisory",
    "https://www.michigan.gov/cra/-/media/Project/Websites/cra/bulletin/1Public-Health-an-Safety-Advisory/Public-Health-and-Safety-Advisory---Marijuana-Consumer-Alert---111522-(002).pdf?rev=5e161e5217be4e92aa3a647108dae628&hash=E894924CA5FE190F404DBCE4E845A344"
  ],
  [
    "09-24-2021 - Notification of Voluntary Marijuana Product Recall",
    "https://www.michigan.gov/cra/-/media/Project/Websites/cra/bulletin/1Public-Health-an-Safety-Advisory/DRAFTMichiganMedicalMarijuanaLLCAUGB000128DBAGLOPublicHealthandSafetyAdvisoryBulletinDHv27366207.pdf?rev=b004a5c780ae4d3c84cfa1238260632a&hash=1618D4238FA861E5BCA9E6B1FDFA7D5D"
  ],
  [
    "07-07-2021 - Notification of Marijuana Product Recall",
    "https://www.michigan.gov/cra/-/media/Project/Websites/cra/bulletin/1Public-Health-an-Safety-Advisory/3843_Euclid_Recall_Notice_7-7-2021_FINAL_729720_7.pdf?rev=7ed2a67934fa43d9a2de8ad259aac5b2&hash=8DF0A6E6A631D955BB672C8A5D2FFE22"
  ],
  [
    "09-17-2020 - Notification of Medical Marijuana Product Recall",
    "https://www.michigan.gov/cra/-/media/Project/Websites/cra/bulletin/1Public-Health-an-Safety-Advisory/9-17-2020_MRA_recall_health_and_safety_bulletin_702606_7.pdf?rev=ed2a117406a6463d99cbff9287b4a55b&hash=26F396B170AA7196D24F2ADD6ABC4C1E"
  ],
  [
    "08-06-2020 - Notification of Voluntary Marijuana Product Recall",
    "https://www.michigan.gov/cra/-/media/Project/Websites/cra/bulletin/1Public-Health-an-Safety-Advisory/3843_Euclid_Recall_Notice_698577_7.pdf?rev=575192df3b56454fa1f281775c754f59&hash=784EFEC8E5AAB407AD4D71894AD19331"
  ],
  [
    "03-20-2020 - Notification of Medical Marijuana Product Recalls",
    "https://www.michigan.gov/cra/-/media/Project/Websites/cra/bulletin/1Public-Health-an-Safety-Advisory/Vape_Carts_Recall_-_03202020_684355_7.pdf?rev=781b71cd09c14e29a23196354f6496f5&hash=B9154BE09B7690E53B88ED1238E72965"
  ],
  [
    "02-07-2020 - Notification of Medical Marijuana Product Recall",
    "https://www.michigan.gov/cra/-/media/Project/Websites/cra/bulletin/1Public-Health-an-Safety-Advisory/Bulletin_-_020620_vitamin_E_recall_680676_7.pdf?rev=6f42e52007e3493a861785147bf708f3&hash=9948E75B39F879A8E18EF75733B5D39A"
  ],
  [
    "01-22-2020 - Notification of Medical Marijuana Product Recall",
    "https://www.michigan.gov/cra/-/media/Project/Websites/cra/bulletin/1Public-Health-an-Safety-Advisory/Plan_BVitaminEPublic_Health_and_Safety_Advisory_01222020_679263_7.pdf?rev=bca9494704d24630b9e9f920d9743a9f&hash=AABE6EEC30086B1CA3868984959F7EAA"
  ],
  [
    "01-17-2020 - Notification of Expanded Marijuana Product Recall",
    "https://www.michigan.gov/cra/-/media/Project/Websites/cra/bulletin/1Public-Health-an-Safety-Advisory/Public_Health_and_Safety_Advisory_Bulletin_678819_7.pdf?rev=e73ec2ae43d9413681e6c790fbd8d1ea&hash=632E62CD93E2858A379F3E6A83C6E185"
  ],
  [
    "01-20-2020 - Notification of Medical Marijuana Flower Recall",
    "https://www.michigan.gov/cra/-/media/Project/Websites/cra/bulletin/1Public-Health-an-Safety-Advisory/Public_Health_and_Safety_Advisory_676863_7.pdf?rev=ffaad15d875041a481593909d2c39c71&hash=A7E23D9814BFB6BB2C67A3BAD1C7A12F"
  ],
  [
    "12-17-2019 - Notification of Multiple Medical Marijuana Products Recall",
    "https://www.michigan.gov/cra/-/media/Project/Websites/cra/bulletin/1Public-Health-an-Safety-Advisory/VitaminEPublic_Health_and_Safety_Advisory_674168_7.pdf?rev=2a3fc07778df4bb4aebaa5a7e66a950d&hash=6313AB525E1CB53B29234B78288F8006"
  ],
  [
    "08-30-2019 - Notification of Medical Marijuana Product Recalls",
    "https://www.michigan.gov/cra/-/media/Project/Websites/cra/bulletin/1Public-Health-an-Safety-Advisory/public-health-safety-advisory-iron-lab-recall-08302019_664856_7.pdf?rev=b638ef9b02dc46729e3f0575c3c13329&hash=75DD83235A1E551A3B35C6D8AD2AFDB7"
  ],
  [
    "08-16-2019 - Notification of Unreliable Safety Compliance Facility Test Results",
    "https://www.michigan.gov/cra/-/media/Project/Websites/cra/bulletin/1Public-Health-an-Safety-Advisory/Iron_Labs_Public_Health_and_Safety_Advisory_Bulletin_081619_663601_7.pdf?rev=48fd02603c0a44419c83059c4d56292c&hash=0228D16D5FD4AEB708597A18660703BC"
  ],
  [
    "08-07-2019 - Notification of Inaccurate Potency Results",
    "https://www.michigan.gov/cra/-/media/Project/Websites/cra/bulletin/1Public-Health-an-Safety-Advisory/The_Spott_Public_Health_and_Safety_Advisory_Bulletin_080719_662637_7.pdf?rev=65b3c1e4670543d09a14c40cf50fce43&hash=8A3958EF552654E9B3B0EB540D81E521"
  ],
  [
    "04-12-2019 - Lead Contamination in Vape Cartridges",
    "https://www.michigan.gov/cra/-/media/Project/Websites/cra/bulletin/1Public-Health-an-Safety-Advisory/Public_Health_and_Safety_Advisory_Lead_Contamination_in_Vape_Cartridges_4-12-2019_652464_7.pdf?rev=6549195f142f469c8c8f72186bec538c&hash=A0B0EFA40DB82B15623E7F06F54D073A"
  ],
  [
    "02-26-2019 - Notification of Multiple Marijuana Products Recall",
    "https://www.michigan.gov/cra/-/media/Project/Websites/cra/bulletin/1Public-Health-an-Safety-Advisory/Public_Health_and_Safety_Advisory_664_Vassar_LLC_Elite_Wellness_Vassar_2-6-2019_003_645544_7.pdf?rev=a0fd4848db594909a577f76615c3b601&hash=6F3AF6ADFFE23686275736FE7C91570A"
  ],
  [
    "02-11-2019 - Notification of Multiple Marijuana Products Recall",
    "https://www.michigan.gov/cra/-/media/Project/Websites/cra/bulletin/1Public-Health-an-Safety-Advisory/Public_Health_and_Safety_Advisory_CCBD_1-10-19_643369_7.pdf?rev=c778c23b3db649e5ba57b6802a7caddb&hash=5624183B2A566D959CB670DA4D9C84F6"
  ],
  [
    "01-04-2019 - Voluntary Recall of Marijuana Products",
    "https://www.michigan.gov/cra/-/media/Project/Websites/cra/bulletin/1Public-Health-an-Safety-Advisory/Public_Health_and_Safety_Advisory_1-4-2019_Final_642608_7.pdf?rev=1d41645ec2ee4765a1fc1530279d1bad&hash=EBCDFE3399C85B3A50F1BE323AF456F6"
  ]
]

def michigan_montana(c):
    url=SITES[c.state];h=dom(c.get(url,"bulletin index",c.state!="Michigan"))
    entries=unique(anchors(h,url))
    section="bulletin index"
    if c.state=="Michigan" and not any(re.match(r"^\d{1,2}[-/]\d{1,2}[-/]\d{4}",label) for label,_ in entries):
        c.audit("bulletin index",url,"review","Live list unavailable. Using the agency bulletin links checked on 2026-11-24; coverage is partial.")
        entries=MI_DOCUMENTS
        section="known official bulletins (partial)"
    for label,link in entries:
        mi=c.state=="Michigan"
        if mi and not re.match(r"^\d{1,2}[-/]\d{1,2}[-/]\d{4}",label):continue
        if not mi and not re.search(r"(?:marijuana|cannabis) product recall",label,re.I):continue
        if mi and re.search(r"first death|requests growers",label,re.I):
            c.audit("bulletin",link,"outside scope",label);continue
        kind="recall" if "recall" in label.lower() else "advisory"
        company=""
        if not mi:
            m=re.search(r"recall\s*[-–—]\s*(.*?)\s*[-–—]\s*\d",label,re.I)
            company=m[1] if m else ""
        e=c.event(urlsplit(link)._replace(query="",fragment="").geturl(),date_in(label),label,link,
                  company=company,kind=kind,section=section)
        inspect_notice(c,e,link)
    if not c.events:c.audit("index",url,"review","No dated product bulletins matched.")

def colorado(c):
    url=SITES[c.state]
    h=dom(c.get(url,"advisory index",False))
    links=[]
    for a in h.select("a[href]"):
        label=txt(a)
        link=urljoin(url,a["href"])
        if not label or not re.search(r"drive\.google|docs\.google|\.pdf(?:$|\?)|advisory",link,re.I):continue
        context=txt(a.find_parent(["li","p","td"]) or a)
        if re.search(r"reporting form|list of stores|sign up|advisory page|recall plan",label,re.I):continue
        if not (date_in(context) or re.search(r"drive\.google|docs\.google|\.pdf",link,re.I)):continue
        if link.rstrip("/")==url.rstrip("/"):continue
        links.append((label,link,date_in(context)))
    if not links:
        c.audit("advisory index",url,"review","Live index unavailable or unrecognized. Using the dated document links recovered from the supplied multistate notebook. Coverage is partial.")
        links=[(title,u,"") for title,u in CO_DOCUMENTS]
        section="saved index links (partial)"
    else:section="live advisory index"
    seen=set()
    for title,link,date in links:
        if file_url(link) in seen:continue
        seen.add(file_url(link))
        e=c.event(file_url(link),date,title,link,kind="health and safety advisory",section=section)
        inspect_notice(c,e,link)

OH_DOCS=[
    ("Paragon Premium Cannabis","https://dam.assets.ohio.gov/image/upload/com.ohio.gov/DCC/2024-04_Product_Recall-_Paragon_Premium_Cannabis.pdf","recall"),
    ("Farmaceutical RX – Local Edibles Ohio","https://dam.assets.ohio.gov/image/upload/com.ohio.gov/DCC/DCC_Product_Alert_FarmaceuticalRX_2025-01.pdf","product alert"),
    ("Standard Farms Ohio","https://com.ohio.gov/divisions-and-programs/cannabis-control/product-safety/SS-Product-Recall","recall"),
]
def ohio(c):
    root=SITES[c.state];links=[]
    for url in [root,root+"/product-safety"]:
        data=c.get(url,"product safety index",False)
        if not data:continue
        h=dom(data)
        for label,u in unique(anchors(h,url)):
            hay=unquote(label+" "+u).replace("_"," ")
            if u.rstrip("/") in {root,root+"/product-safety"}:continue
            if not re.search(r"recall|product.alert",hay,re.I) or re.search(r"template|guidance|newsletter|policy|rule package|faq",hay,re.I):continue
            links.append((label or unquote(urlsplit(u).path.rsplit("/",1)[-1]),u,"product alert" if not re.search("recall",hay,re.I) else "recall"))
        if links:break
    section="product safety index"
    if not links:
        c.audit("product safety index",root,"review","The live index could not be read. Only the individually identified DCC notices are attempted.")
        links=OH_DOCS
        section="known official notices (partial)"
    for title,link,kind in unique(links):
        e=c.event(link,date_in(title),title,link,kind=kind,section=section)
        inspect_notice(c,e,link)
        if link==OH_DOCS[2][1] and not e["recall_date"] and not e["recall_reason"]:
            # Dated agency notice metadata for the listing returning 404, checked 2026-11-24.
            # Keep the group-level record; do not manufacture forty individual product names.
            e.update(recall_date="2026-04-21",date_basis="agency notice date (saved metadata)",
                     company="Standard Farms Ohio LLC",scope="Cannabis-infused gummy products (40 products in the notice)",
                     recall_reason="The gummy labels omitted soy oil and soy lecithin.",
                     source_section="known official notice metadata (partial)")
            c.audit("product list",link,"review","Only notice-level metadata is available until the agency table can be retrieved.")

## 7. Washington

Each dated block can contain product names, identifiers, a producer and a reason. Read the linked spreadsheet when the page lists its affected products there.

In [10]:
def state_lines(node,base):
    for a in node.find_all("a",href=True):
        a.replace_with("⟦"+urljoin(base,a["href"])+"⟧"+txt(a)+"⟦/⟧")
    for n in node.find_all(string=True):
        # Keep inline bold labels and the values next to them on the same line.
        if "\n" in str(n):n.replace_with(re.sub(r"\s*\n\s*"," ",str(n)))
    for el in node.find_all("br"):el.replace_with("\n")
    for el in node.find_all(["p","div","li","h2","h3","h4","section","tr","td","hr"]):
        el.insert_before("\n");el.append("\n")
    return [clean(x) for x in node.get_text("").splitlines() if clean(x)]

WA_LINK=re.compile(r"⟦([^⟧]+)⟧(.*?)⟦/⟧")
WA_ID=re.compile(r"\b(?:[0-9a-f]{8}-(?:[0-9a-f]{4}-){3}[0-9a-f]{12}|(?=[a-z\d.]*\d{4})[a-z\d.]{10,})\b",re.I)
def washington(c):
    url=SITES[c.state];h=dom(c.get(url,"recall index"))
    entries=[];cur=None;field=""
    for raw in state_lines(content(h),url):
        line=WA_LINK.sub(lambda m:m[2],raw)
        m=re.match(r"^Date\s*:\s*(.*)",line,re.I)
        if m:
            cur={"date":date_in(m[1]),"products":[],"producer":[],"reason":[],"kind":[],"files":[]};entries.append(cur);field="";continue
        if cur is None:continue
        if re.fullmatch(r"\d{4}(?:\s*-\s*\d{4})?",line):field="";continue
        m=re.match(r"^(Products?|Producer\s*/\s*Processor|Address|Type of Recall|Reason(?: for Recall)?)\s*:\s*(.*)",line,re.I)
        if m:
            k=key(m[1]);line=m[2]
            field="producer" if k.startswith("producer") else "products" if k.startswith("product") else "reason" if k.startswith("reason") else "kind" if k=="typeofrecall" else ""
        files=[u for u,t in WA_LINK.findall(raw) if re.search(r"\.(?:xlsx?|pdf|csv)(?:\?|$)",u,re.I)]
        cur["files"].extend(files)
        if field and line and not (files and field=="products"):cur[field].append(line)
    for i,b in enumerate(entries):
        company=" ".join(b["producer"])
        reason=" ".join(b["reason"])
        if key(reason) in {"na","none"}:reason=""
        title="Recall – "+company if company else "Cannabis product recall"
        ident="|".join([url,b["date"],company," ".join(b["products"][:2])," ".join(b["files"]),reason[:120]])
        e=c.event(ident,b["date"],title,url,reason=reason,company=company,kind=" ".join(b["kind"]) or "recall",section="recall index")
        for line in b["products"]:
            name=WA_ID.sub("",line).strip(' –—-,:"“”')
            if not name or key(name) in {"na","none","productid","productids","retailids","lotnumbers"}:continue
            if re.fullmatch(r"[#\d /,.-]+",name) or re.fullmatch(r"(?:see|refer to).*?(?:sheet|link|list|attachment).*",name,re.I):continue
            level="product group" if re.search(r"^(?:certain products|all products|cannabis materials|mj flower)$",name,re.I) else "index product description"
            c.add(e,name,url,level=level)
        for u in unique(b["files"]):
            if ".pdf" in u.lower():pdf_products(c,e,u,attachment=True)
            else:spreadsheet(c,e,u)
    if not entries:c.audit("index",url,"review","No Date blocks matched.")

## 8. Write the CSVs and workbooks

The **Notices** worksheet has one row per source notice. **Recalls** contains the associated product or scope rows. **Coverage** summarizes the state. **Sources** records retrievals and review items. Use the CSV’s `review` field to find unresolved dates, reasons or product descriptions.

The notebook builds each state’s files before replacing an earlier result. On a rerun, the prior state folder moves to `recall_outputs/_previous/<run timestamp>/`. Close any open output workbooks before rerunning on Windows.

In [11]:
from tempfile import TemporaryDirectory
import os
import uuid
import xlsxwriter

SCRAPERS={"California":california,"Colorado":colorado,"Missouri":missouri,"Maine":maine,
          "Michigan":michigan_montana,"Montana":michigan_montana,"New Jersey":new_jersey,
          "New Mexico":new_mexico,"New York":new_york,"Ohio":ohio,"Washington":washington}
def write_workbook(path,sheets):
    # Source text and product identifiers stay literal, including any leading zeroes.
    with pd.ExcelWriter(path,engine="xlsxwriter",engine_kwargs={"options":{"strings_to_urls":False,"strings_to_formulas":False}}) as writer:
        for sheet,df in sheets.items():
            df.to_excel(writer,sheet_name=sheet,index=False)
            ws=writer.sheets[sheet];ws.freeze_panes(1,0)
            ws.autofilter(0,0,max(1,len(df)),len(df.columns)-1)
            for i,col in enumerate(df.columns):
                width=max([len(col),12]+[len(str(s)) for s in df[col].head(100)])
                ws.set_column(i,i,min(70,width+2))

def run_state(state,stamp):
    c=Collection(state);target=c.folder
    with TemporaryDirectory(dir=OUTPUT,prefix=".building-") as tmp:
        c.folder=Path(tmp)/target.name;c.folder.mkdir()
        try:
            SCRAPERS[state](c)
        except Exception as exc:
            c.audit("state collection",SITES[state],"review",f"{type(exc).__name__}: {exc}. Results collected before this error were retained.")
        finally:
            c.session.close()
        data,log=c.frames()
        notices=pd.DataFrame(list(c.events.values()),columns=["recall_id","state","recall_date","date_basis","notice_title","notice_type",
                "source_section","company","recall_reason","scope","notice_url"]).fillna("")
        if data.empty and not (log["result"]=="review").any():
            c.audit("state collection",SITES[state],"review","No individual notices were extracted.");data,log=c.frames()
        field_issues=int(data["review"].ne("").sum())
        source_issues=int(log["result"].eq("review").sum())
        coverage=("no records extracted" if not len(data) else "partial index" if
                  any("partial" in v.casefold() for v in notices["source_section"]) else
                  "review" if field_issues or source_issues else "collected")
        row=dict(state=state,notices=len(notices),product_or_scope_rows=len(data),rows_to_review=field_issues,
                 source_review_items=source_issues,coverage=coverage,run_utc=stamp)
        data.to_csv(c.folder/"recalls.csv",index=False,encoding="utf-8-sig")
        write_workbook(c.folder/"recalls.xlsx",{"Recalls":data,"Notices":notices,"Coverage":pd.DataFrame([row]),"Sources":log})
        # Keep earlier results, including data from a more successful past run.
        backup=OUTPUT/"_previous"/stamp/target.name
        if target.exists():
            backup.parent.mkdir(parents=True,exist_ok=True)
            target.rename(backup)
        try:c.folder.rename(target)
        except OSError:
            if backup.exists() and not target.exists():backup.rename(target)
            raise
    return row

def run_collection(states):
    states=list(dict.fromkeys(states))
    invalid=set(states)-set(SCRAPERS)
    if invalid or not states:raise ValueError(f"Select one or more state names from SITES. Unknown names: {invalid}")
    OUTPUT.mkdir(parents=True,exist_ok=True)
    stamp=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"-"+uuid.uuid4().hex[:6]
    summary_path=OUTPUT/"collection_summary.csv"
    prior=pd.read_csv(summary_path,keep_default_na=False) if summary_path.exists() else pd.DataFrame()
    rows=[]
    for state in states:
        print(f"Collecting {state} …",flush=True)
        rows.append(run_state(state,stamp))
        row=rows[-1]
        print(f"  {row['notices']} notices • {row['product_or_scope_rows']} rows • {row['coverage']}",flush=True)
        current=pd.DataFrame(rows)
        if not prior.empty and "state" in prior.columns:
            current=pd.concat([prior[~prior["state"].isin(current["state"])],current],ignore_index=True)
        tmp=OUTPUT/(stamp+".csv.tmp")
        current.sort_values("state").to_csv(tmp,index=False,encoding="utf-8-sig")
        os.replace(tmp,summary_path)
    return pd.DataFrame(rows)

## 9. Run the collection

In [12]:
summary = run_collection(STATES_TO_RUN)
display(summary)
print("Results:", OUTPUT.resolve())

  California portal page 10 …
  California portal page 20 …
  California portal page 30 …
  200 notices • 894 rows • review
Consider using the pymupdf_layout package for a greatly improved page layout analysis.
  38 notices • 426 rows • partial index
  14 notices • 7563 rows • review
  11 notices • 53 rows • collected
  30 notices • 177 rows • review
  2 notices • 6 rows • collected
  2 notices • 2 rows • collected
  4 notices • 50 rows • review
  3 notices • 93 rows • collected
  3 notices • 79 rows • partial index
  43 notices • 110 rows • review


,state,notices,product_or_scope_rows,rows_to_review,source_review_items,coverage,run_utc
0,California,200,894,7,10,review,20261124T213024Z-79618e
1,Colorado,38,426,5,39,partial index,20261124T213024Z-79618e
2,Missouri,14,7563,0,1,review,20261124T213024Z-79618e
3,Maine,11,53,0,0,collected,20261124T213024Z-79618e
4,Michigan,30,177,5,30,review,20261124T213024Z-79618e
5,Montana,2,6,0,0,collected,20261124T213024Z-79618e
6,New Jersey,2,2,0,0,collected,20261124T213024Z-79618e
7,New Mexico,4,50,1,1,review,20261124T213024Z-79618e
8,New York,3,93,0,0,collected,20261124T213024Z-79618e
9,Ohio,3,79,1,5,partial index,20261124T213024Z-79618e


Results: C:\Users\pegah\Downloads\recall_outputs


## 10. Inspect a state

Start with the product, date and reason columns. Use the notice ID to look up related records. The last table displays source problems and documented empty sections. Review **partial index** states before comparing state totals. Count distinct `recall_id` values for notices, since one notice can list hundreds of products.

In [13]:
state_to_view = "Michigan" if "Michigan" in STATES_TO_RUN else STATES_TO_RUN[0]
folder = OUTPUT / state_to_view.lower().replace(" ", "_")
products = pd.read_csv(folder / "recalls.csv", dtype=str, keep_default_na=False)
sources = pd.read_excel(folder / "recalls.xlsx", sheet_name="Sources", dtype=str, keep_default_na=False)
display(products[["recall_date","product_name","recall_reason","notice_type","record_level","product_status","notice_url"]].head(20))
display(sources.loc[~sources["result"].isin(["ok","outside scope"])])

,recall_date,product_name,recall_reason,notice_type,record_level,product_status,notice_url
0,2025-08-27,Kushy Punch-Vapes (Pineapple Jealousy),"In its investigation, the CRA discovered throu...",recall,named product,listed,https://www.michigan.gov/cra/-/media/Project/W...
1,2025-08-12,Motor City Cannacarts – Confetti Cake,Various flavors of Motor City Cannacarts and R...,recall,named product,listed,https://www.michigan.gov/cra/-/media/Project/W...
2,2025-08-12,Motor City Cannacarts – Georgia Pie,Various flavors of Motor City Cannacarts and R...,recall,named product,listed,https://www.michigan.gov/cra/-/media/Project/W...
3,2025-08-12,RIPZ – Ice Cream Man,Various flavors of Motor City Cannacarts and R...,recall,named product,listed,https://www.michigan.gov/cra/-/media/Project/W...
4,2025-08-12,RIPZ – Peach Cobbler,Various flavors of Motor City Cannacarts and R...,recall,named product,listed,https://www.michigan.gov/cra/-/media/Project/W...
5,2025-08-12,RIPZ – Tropic Thunder,Various flavors of Motor City Cannacarts and R...,recall,named product,listed,https://www.michigan.gov/cra/-/media/Project/W...
6,2025-08-12,RIPZ – Wedding Cake,Various flavors of Motor City Cannacarts and R...,recall,named product,listed,https://www.michigan.gov/cra/-/media/Project/W...
7,2025-02-11,Alien Jack,Various flavors of BLOOM Classic and BLOOM Liv...,recall,named product,listed,https://www.michigan.gov/cra/-/media/Project/W...
8,2025-02-11,Blue Dream,Various flavors of BLOOM Classic and BLOOM Liv...,recall,named product,listed,https://www.michigan.gov/cra/-/media/Project/W...
9,2025-02-11,Cereal Milk,Various flavors of BLOOM Classic and BLOOM Liv...,recall,named product,listed,https://www.michigan.gov/cra/-/media/Project/W...


,state,kind,url,result,detail
2,Michigan,PDF extraction,https://www.michigan.gov/cra/-/media/Project/W...,review,"Check the notice for additional products, lot-..."
4,Michigan,PDF extraction,https://www.michigan.gov/cra/-/media/Project/W...,review,"Check the notice for additional products, lot-..."
6,Michigan,PDF extraction,https://www.michigan.gov/cra/-/media/Project/W...,review,"Check the notice for additional products, lot-..."
8,Michigan,PDF extraction,https://www.michigan.gov/cra/-/media/Project/W...,review,"Check the notice for additional products, lot-..."
10,Michigan,PDF extraction,https://www.michigan.gov/cra/-/media/Project/W...,review,"Check the notice for additional products, lot-..."
12,Michigan,PDF extraction,https://www.michigan.gov/cra/-/media/Project/W...,review,"Check the notice for additional products, lot-..."
14,Michigan,PDF extraction,https://www.michigan.gov/cra/-/media/Project/W...,review,"Check the notice for additional products, lot-..."
16,Michigan,PDF extraction,https://www.michigan.gov/cra/-/media/Project/W...,review,"Check the notice for additional products, lot-..."
18,Michigan,PDF extraction,https://www.michigan.gov/cra/-/media/Project/W...,review,"Check the notice for additional products, lot-..."
20,Michigan,PDF extraction,https://www.michigan.gov/cra/-/media/Project/W...,review,"Check the notice for additional products, lot-..."


### Recover a blocked source

Open the URL from **Sources** in your browser. If it opens, save the PDF or spreadsheet, or save an index as **Webpage, HTML Only** with its links intact. For an embedded table, save the page containing the table. In section 2 map the original URL to that local file, for example `LOCAL_SOURCES = {SITES["Colorado"]: Path("saved/colorado_index.html")}`. You can map individual PDF URLs in the same dictionary. Rerun section 2 and section 9 with the state you want to refresh. If an agency page remains unavailable, the state summary and the previously published records are available for review.